4장에서 수치 미분을 통해서 가중치 매개변수의 기울기를 구하였다. 단순하지만 오래 걸린다. 그래서 효율적으로 계산하는 오차역전파법을 배우자

## 계산 그래프 
계산 과정을 그래프로 나타낸 것이다.

### 국소적 계산
계산 그래프는 국소적 계산을 전파하여 최종 결과를 얻는다.

### 왜 계산 그래프로 푸는가?
이점은 국소적 계산이라는 것이다. 그리고 중간 계산 결과를 모두 보관할 수 있다. 또 이것 뿐만이 아니라 역전파를 통해서 미분을 효율적으로 계산할 수 있다.  

## 연쇄법칙
역전파는 국소적인 미분을 오른쪽에서 왼쪽으로 전달한다. 이런 국소적 미분을 전달하는 원리는 연쇄법칙에 따른 것이다. 그리고 연쇄법칙은 계산 그래프 상에서 역전파와 같다.

### 계산 그래프의 역전파
역전파의 계산 절차는 신호에 노드의 국소적 미분을 곱한 후 다음 노드에 전달하는 것이다. 여기서 국소적 미분은 순전파 함수의 미분이다. 그리고 연쇄법칙에 의해서 국소적 미분계수?를 곱해서 결론에 이르게 한다.

## 역전파

### 덧셈 노드의 역전파
1을 곱하기만 할 뿐 입력된 값을ㄹ 그대로 다음 노드에 보낸다. x + y = z(x, y), dz/dx = 1, dz/dy = 1이기 때문에 입력인 df/dz = df/dz dz/dy = df/dz * 1, y에 대해서도 같다.

### 곱셈 노드의 역전파
상류의 값에 순전파 때의 입력 신호들을 서로 바꾼 값을 곱해서 하류로 보낸다. xy = z(x,y), dz/dx = y, dz/dy = x이기 때문이다.

In [3]:
import numpy as np

In [1]:
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None
    
    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y
        
        return out

    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x

        return dx, dy

apple = 100
apple_num = 2
tax = 1.1

mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

apple_price = mul_apple_layer.forward(apple, apple_num)
price = mul_tax_layer.forward(apple_price, tax)

print(price)

dprice = 1
dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)

print(dapple, dapple_num, dtax)

220.00000000000003
2.2 110.00000000000001 200


In [ ]:
class AddLayer:
    def __init__(self):
        pass

    def forward(self, x, y):
        out = x + y
        return out

    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        return dx, dy

## 활성화 함수 계층 구현하기
신경망을 구성하는 계층을 각각을 클래스 하나로 구현하자.  

### ReLU 계층
ReLU 수식은 이와 같다. y = {x (x >0), 0 (x<= 0)}, dy/dx = {1 (x >0), 0 (x<= 0)} 

In [4]:
x = np.array([[1.0, -.5], [-2.0, 3.0]])
print(x)

mask = (x <= 0)
print(mask)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]


### Sigmoid 계층
Sigmoid 함수는 1/(1 + exp(-x)) 인데 이를 어떻게 표현할까? 바로 국소적 미분을 이용하는 것이다.

### Affine/Softmax 계층
신경망의 순전파에서는 가중치 신호의 총합을 계산하므로 행렬의 곱을 이용한다. 3장을 참고하자. 기하학에서는 이런 행렬의 곱을 어파인 변환이라고 한다. 그럼 이 과정을 역전파로 어떻게 바꿀까? 편향은 덧셈으로 연결되어 있어 전에 해결한 적이 있지만 곱은 어떻게 국소적 미분을 적용하냐면 전치를 시키면 된다.
그래서 출력으로 dL/dX = dL/dY * W^T, dL/dW = X^T * dL/dY이다.

소프트맥스 계층은 교차 엔트로피 오차 계층과 함께 구현한다. 순전파에 경우에 소프트맥스에서 출력이 정답 레이블을 받아 오차 계층으로 이동하여 손실이 판단된다. 이의 역전파는 어떨까? 놀랍게도 출력 - 정답 레이블의 벡터로 나온다는 것이다. 그래서 손실이 앞 계층으로 전달되는 것이다. 

## 오차역전파법으로 신경망 구현, 기울기 검증, 학습 구현

In [6]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
from functions import *
from gradient import *
from layers import *
from collections import OrderedDict

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=.01):
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])
        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])

        return accuracy

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads

    def gradient(self, x, t):
        self.loss(x, t)

        dout = 1
        dout = self.lastLayer.backward(dout)
        
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)
        
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db

        return grads
        

In [8]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
from dataset.mnist import load_mnist
from two_layer_net5 import TwoLayerNet

(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

for key in grad_numerical.keys():
    diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))
    print(key + ":" + str(diff))

W1:4.597503742738479e-10
b1:2.7813424266129838e-09
W2:5.274147278913285e-09
b2:1.4033391261547568e-07


In [10]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
from dataset.mnist import load_mnist

(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 20000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = .1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    grad = network.gradient(x_batch, t_batch)

    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]
    
    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)

0.1026 0.107
0.9049666666666667 0.9089
0.9219833333333334 0.9257
0.9340833333333334 0.9335
0.9447333333333333 0.943
0.9522166666666667 0.9506
0.95675 0.9544
0.9618833333333333 0.9611
0.9637666666666667 0.9594
0.9666833333333333 0.9608
0.9686 0.9627
0.9708666666666667 0.9661
0.9733166666666667 0.965
0.9747833333333333 0.9666
0.9761 0.9692
0.9775 0.9697
0.97805 0.9706
0.9790333333333333 0.9708
0.9800833333333333 0.9708
0.9802833333333333 0.9705
0.9818666666666667 0.9718
0.9820833333333333 0.9719
0.9836166666666667 0.9719
0.9819333333333333 0.9711
0.9840166666666667 0.9717
0.9849333333333333 0.9712
0.98645 0.9735
0.9869833333333333 0.9742
0.9881 0.9737
0.98695 0.9729
0.9885166666666667 0.9725
0.98795 0.9737
0.9893333333333333 0.9734
0.9900166666666667 0.975
